# 7. Figures

Redraws the paper's figures:

| figure | paper | needs |
|---|---|---|
| whole-tumor segmentation on three cases | `fig:segmentation` | the BraTS copy and `brats_seg_cf/` |
| label-free sub-region estimation on two cases | `fig:subregions` | the BraTS copy and `brats_seg_cf/` |
| SHAP attributions for grading | `fig:shap` | `data_cache/` only |

The first two draw the images themselves, so they need the BraTS working copy and the
cross-fitted masks. The third runs from the cached features. All three need `matplotlib`, and the
third also needs `shap`.

The palette is a colour-vision-deficiency-safe set that stays distinguishable in every pair. Blue
and orange separate the two grades; the enhancing, necrotic and edema triad reuses the same three,
so a reader who has learned one figure can read the next.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from src import config
print(config.describe())

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from src import figures, segmentation, subregions

figures.set_style()
HAVE_VOLUMES = config.NPZ_DIR.exists() and config.SEG_DIR.exists()
print("volumes and cross-fitted masks present:", HAVE_VOLUMES)

## fig:segmentation

Three rows: a high-grade case, a low-grade case, and the weakest case in the cohort by Dice
(BraTS20_Training_087, see notebook 1), so the figure is not only its best cases. Columns: T1ce,
the reference whole tumor, the predicted whole tumor, and the two compared, split into overlap,
tissue the prediction missed and tissue predicted beyond the reference.

The slice is chosen on the **reference** mask, so the row is not picked to flatter the prediction,
and the Dice quoted is over the whole volume rather than that slice. What the prediction misses
lies at the edema margin, not on the enhancing core.

In [ ]:
SEGMENTATION_CASES = [
    ("BraTS20_Training_045", "High-grade (HGG)"),
    ("BraTS20_Training_291", "Low-grade (LGG)"),
    ("BraTS20_Training_087", "Worst case (HGG)"),
]

if HAVE_VOLUMES:
    fig, axes = plt.subplots(3, 4, figsize=(7.0, 5.5))
    titles = ["T1ce", "Ground truth", "Predicted (MedSAM2)", "Prediction vs.\nground truth"]

    for r, (case, tag) in enumerate(SEGMENTATION_CASES):
        volumes = figures.load_case(case)
        reference, predicted = volumes["seg"] > 0, volumes["wt"]
        volumetric_dice = figures.dice(predicted, reference)

        z = figures.largest_slice(reference)
        ref2d, pred2d = reference[z], predicted[z]
        rows, cols = figures.crop_bbox(ref2d | pred2d)

        gray = figures.window(volumes["t1ce"][z])[rows, cols]
        ref2d, pred2d = ref2d[rows, cols], pred2d[rows, cols]
        overlap = ref2d & pred2d
        missed = ref2d & ~pred2d
        extra = pred2d & ~ref2d

        axes[r, 0].imshow(gray, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        figures.paint(axes[r, 1], gray, [(ref2d, figures.MISSED_COLOR)])
        figures.paint(axes[r, 2], gray, [(pred2d, figures.EXTRA_COLOR)])
        figures.paint(axes[r, 3], gray, [(overlap, figures.OVERLAP_COLOR),
                                         (missed, figures.MISSED_COLOR),
                                         (extra, figures.EXTRA_COLOR)])
        axes[r, 3].text(0.97, 0.04, f"Dice {volumetric_dice:.2f}",
                        transform=axes[r, 3].transAxes, fontsize=7.5, color=figures.INK,
                        ha="right", va="bottom",
                        bbox=dict(boxstyle="round,pad=0.22", fc="white",
                                  ec=figures.GRID, lw=0.6))
        for c in range(4):
            figures.bare(axes[r, c])
        axes[r, 0].set_ylabel(tag, fontsize=9.5, color=figures.INK, labelpad=6)

    for c in range(4):
        axes[0, c].set_title(titles[c], color=figures.INK, pad=6)

    fig.legend(handles=[
        Patch(facecolor=figures.OVERLAP_COLOR, edgecolor="none", alpha=0.7, label="Overlap"),
        Patch(facecolor=figures.MISSED_COLOR, edgecolor="none", alpha=0.7,
              label="Missed by the prediction"),
        Patch(facecolor=figures.EXTRA_COLOR, edgecolor="none", alpha=0.7,
              label="Predicted beyond the ground truth"),
    ], loc="lower center", ncol=3, handlelength=1.1, handleheight=1.1, borderpad=0.2,
        columnspacing=1.6, bbox_to_anchor=(0.5, 0.005))
    fig.subplots_adjust(left=0.055, right=0.99, top=0.925, bottom=0.075,
                        wspace=0.06, hspace=0.08)
    figures.save(fig, "fig_segmentation.png")
    plt.show()

## fig:subregions

Two rows, a high-grade and a low-grade case. Columns: T1ce with the predicted whole-tumor contour;
the enhancement map in standard deviations of healthy brain; and the estimated enhancing, necrotic
and edema sub-regions.

The high-grade tumor enhances broadly while the low-grade one does not, and the compartments
follow from the two thresholds of notebook 2 with no reference label involved.

In [ ]:
SUBREGION_CASES = [("BraTS20_Training_045", "High-grade (HGG)"),
                   ("BraTS20_Training_291", "Low-grade (LGG)")]

if HAVE_VOLUMES:
    fig, axes = plt.subplots(2, 3, figsize=(7.0, 4.9))
    titles = ["T1ce + predicted tumour",
              "Enhancement map  (T1ce $-$ T1, $\\sigma$)",
              "Estimated sub-regions"]

    for r, (case, tag) in enumerate(SUBREGION_CASES):
        volumes = figures.load_case(case)
        z = figures.largest_slice(volumes["wt"])
        t1ce, t1, flair = volumes["t1ce"][z], volumes["t1"][z], volumes["flair"][z]
        wt = volumes["wt"][z]
        ET, NCR, ED = subregions.estimate_subregions(wt, t1ce, t1, flair)
        rows, cols = figures.crop_bbox(wt)

        gray = figures.window(t1ce)[rows, cols]
        contour = wt[rows, cols]
        sigma = subregions.enhancement_sigma(t1ce, t1, wt, flair)[rows, cols]
        masks = {k: m[rows, cols] for k, m in (("ET", ET), ("NCR", NCR), ("ED", ED))}

        ax = axes[r, 0]
        ax.imshow(gray, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        ax.contour(contour, levels=[0.5], colors=[figures.INK], linewidths=1.1)
        ax.contour(contour, levels=[0.5], colors=["white"], linewidths=0.4)

        image = axes[r, 1].imshow(np.clip(sigma, -1, 6), cmap=figures.ENHANCEMENT_CMAP,
                                  interpolation="nearest")

        # Edema first, enhancing on top, so the small compartment stays visible.
        figures.paint(axes[r, 2], gray,
                      [(masks[k], figures.SUB_COLORS[k]) for k in ("ED", "NCR", "ET")],
                      alpha=0.62)

        for c in range(3):
            figures.bare(axes[r, c])
        axes[r, 0].set_ylabel(tag, fontsize=9.5, color=figures.INK, labelpad=6)

    for c in range(3):
        axes[0, c].set_title(titles[c], color=figures.INK, pad=6)

    bar = fig.colorbar(image, ax=axes[:, 1], location="bottom", fraction=0.046,
                       pad=0.04, aspect=32)
    bar.set_label("$\\sigma$ above healthy brain", fontsize=7.5, color=figures.INK_SECONDARY)
    bar.ax.tick_params(labelsize=7, color=figures.MUTED)
    bar.outline.set_visible(False)

    axes[1, 2].legend(handles=[
        Patch(facecolor=figures.SUB_COLORS[k], edgecolor="none", alpha=0.75,
              label=figures.SUB_LABELS[k]) for k in ("ET", "NCR", "ED")
    ], loc="upper center", bbox_to_anchor=(0.5, -0.04), ncol=1,
        handlelength=1.1, handleheight=1.1, borderpad=0.2, labelspacing=0.35)

    fig.subplots_adjust(left=0.06, right=0.99, top=0.93, bottom=0.16,
                        wspace=0.06, hspace=0.10)
    figures.save(fig, "fig_subregions.png")
    plt.show()

## fig:shap -- what carries the grade

A random forest with the protocol's settings (300 trees, depth 6, balanced classes, seed 42) is
fit on all 369 cases and the seventeen cross-fitted features, and a tree explainer attributes its
low-grade probability to the features. The figure shows the twelve strongest, in matplotlib's
default style. The strongest features measure the estimated enhancing sub-region (its fraction,
its volume and its mean T1ce intensity), so the grading signal comes from the sub-regions and not
from texture, of which the features hold none.

The attributions are deterministic. The plot draws its dots in a random order, so the image
matches the paper's in content rather than pixel for pixel.

In [ ]:
try:
    import shap
    HAVE_SHAP = True
except ImportError:
    HAVE_SHAP = False
    print("shap is not installed; install it to draw fig:shap")

if HAVE_SHAP:
    from src import cohort, evaluation

    X, y, cases = cohort.build_matrix(cohort.extract_features(), cohort.load_grade_labels())
    names = [figures.FEATURE_DISPLAY_NAMES[n] for n in subregions.FEATURE_NAMES]
    forest = evaluation.CLASSIFIERS["Random forest"]().fit(X, y)
    values = shap.TreeExplainer(forest).shap_values(X)
    # A binary forest gives one matrix per class; the low-grade side is the positive class.
    values = values[1] if isinstance(values, list) else (
        values[..., 1] if np.ndim(values) == 3 else values)

    ranking = pd.Series(np.abs(values).mean(0), index=names).sort_values(ascending=False)
    display(ranking.round(4).to_frame("mean |SHAP|"))

    with plt.style.context("default"):
        # The beeswarm shuffles the dot order and jitters it with numpy's global generator, so
        # it is seeded here; the attributions themselves are deterministic.
        np.random.seed(0)
        plt.figure()
        shap.summary_plot(values, X, feature_names=names, show=False, max_display=12)
        plt.tight_layout()
        config.FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        plt.savefig(config.FIGURE_DIR / "shap_grade.png", dpi=150, bbox_inches="tight")
        print(f"wrote {config.FIGURE_DIR / 'shap_grade.png'}")
        plt.show()